In [5]:
import numpy as np
import pandas as pd
import nflreadpy as nfl

Essentially, Claude determined that much of return yards are randomness, especially game to game. Player skill specifically is irrelevant, but the most predictive thing is how many returns the team gets, what the share of the teams returns a player gets, and a small multiplier for matchup (specifically that touchback rate is pretty predictive)

In [29]:
seasons = [2023, 2024, 2025]

In [40]:
def load_return_plays(seasons):
    """
        Return all punt and kickoff plays from play-by-play
    """
    pbp = nfl.load_pbp(seasons = seasons).to_pandas()

    st = pbp[pbp.play_type.isin(["punt", "kickoff"])].copy()

    # Add universal returner id and returner name
    st["returner_id"] = st.punt_returner_player_id.fillna(st.kickoff_returner_player_id)
    st["returner_name"] = st.punt_returner_player_name.fillna(st.kickoff_returner_player_name)

    is_kickoff = st.play_type.eq("kickoff")
    st["return_team"] = np.where(is_kickoff, st.posteam, st.defteam)
    st["kick_team"] = np.where(is_kickoff, st.defteam, st.posteam)

    # Mark whether there is a return on the play
    fair_caught = st.punt_fair_catch.fillna(0).eq(1)
    st["is_return"] = st.returner_id.notna() & ~fair_caught

    return st[["game_id", "season", "week", "play_type", "kick_team", "return_team",
               "returner_id", "returner_name", "is_return", "return_yards", "touchdown"]]

In [43]:
plays = load_return_plays(seasons)
plays.head()

,game_id,season,week,play_type,kick_team,return_team,returner_id,returner_name,is_return,return_yards,touchdown
1,2023_01_ARI_WAS,2023,1,kickoff,ARI,WAS,None,None,False,0.0,0.0
10,2023_01_ARI_WAS,2023,1,punt,WAS,ARI,None,None,False,0.0,0.0
17,2023_01_ARI_WAS,2023,1,punt,ARI,WAS,00-0031941,J.Crowder,True,0.0,0.0
27,2023_01_ARI_WAS,2023,1,kickoff,WAS,ARI,None,None,False,0.0,0.0
37,2023_01_ARI_WAS,2023,1,kickoff,ARI,WAS,None,None,False,0.0,0.0


In [42]:
# Sanity check -- every busy returner should show exactly one return_team.
chk = (plays[plays.is_return & plays.season.eq(2025)]
       .groupby("returner_id")
       .agg(n=("is_return", "size"), teams=("return_team", "nunique")))
print(chk[chk.n >= 15].teams.value_counts())   # expect all 1s

teams
1    73
2     3
Name: count, dtype: int64


In [ ]:
def build_returner_usage(plays, season=2025, through_week=None):
    """
        Create DataFrame summarizing player by player return share, expected returns
    """
    d = plays[plays.season == season]
    if through_week is not None:
        d = d[d.week <= through_week]

    frames = []
    for kind, tag in [("kickoff", "ko"), ("punt", "pr")]: # for both kickoffs and punts
        sub = d[d.play_type == kind]

        # Calculate Team Returns Per Game
        games = sub.groupby("return_team").game_id.nunique()
        team_ret = sub.groupby("return_team").is_return.sum()
        team_rate = team_ret / games

        # Calculate Individual Player Return Share, and Expected Returns
        r = sub[sub.is_return]
        u = r.groupby(["returner_id", "returner_name", "return_team"]).size().rename("n").reset_index()
        u[f"{tag}_share"] = u.n / u.return_team.map(team_ret)
        u[f"exp_{tag}_returns"] = u.return_team.map(team_rate) * u[f"{tag}_share"]

        frames.append(u.drop(columns="n").set_index(["returner_id", "returner_name", "return_team"]))

    # Output Total DataFrame for Punts, Kickoffs, and Total
    out = frames[0].join(frames[1], how="outer").fillna(0.0).reset_index()
    out["exp_total_returns"] = out.exp_ko_returns + out.exp_pr_returns
    return out.sort_values("exp_total_returns", ascending=False).reset_index(drop=True)


In [45]:
usage = build_returner_usage(plays, season=2025)
usage.head(5)

,returner_id,returner_name,return_team,ko_share,exp_ko_returns,pr_share,exp_pr_returns,exp_total_returns
0,00-0040506,M.Price,MIN,0.850746,3.352941,0.937500,1.875000,5.227941
1,00-0040705,C.Dike,TEN,0.953846,3.647059,0.958333,1.437500,5.084559
2,00-0037801,K.Turpin,DAL,0.831325,4.058824,0.769231,0.588235,4.647059
3,00-0036331,D.Duvernay,CHI,0.608108,2.368421,1.000000,1.500000,3.868421
4,00-0040138,J.Noel,HOU,0.573770,1.842105,0.925000,1.947368,3.789474


In [46]:
def build_yardage_pools(plays, seasons=(2025,)):
    """
        Collect real NFL returns to draw from, separated by kick type

        Returns dict mapping 'kickoff'/'punt' to a (yards, touchdowns) array pair
    """

    r = plays[plays.is_return & plays.season.isin(seasons) & plays.return_yards.notna()]
    return {k: (d.return_yards.to_numpy(float), d.touchdown.fillna(0).to_numpy(float))
            for k, d in r.groupby("play_type")}

def opponent_multipliers(plays, season=2025, through_week=None, shrink=0.5):
    """
        Return a mapping of team abbreviation to a multiplier representative
        of how many kickoff returns they handoff. This comes from obtaining a ratio
        of how many kickoff returns are allowed by the team per game against league
        average, and shrinks the ratio (shrink)% of the way towards 1.0 (league average).
    """

    # Get Kickoff Plays
    d = plays[(plays.season == season) & plays.play_type.eq("kickoff")]
    if through_week is not None:
        d = d[d.week <= through_week]

    allowed = d.groupby(["kick_team", "game_id"]).is_return.sum().groupby("kick_team").mean()
    ratio = allowed / allowed.mean()
    return 1.0 + shrink * (ratio - 1.0)

In [47]:
pools = build_yardage_pools(plays, seasons=(2025,))
opp_mult = opponent_multipliers(plays, season=2025)
opp_mult.sort_values()   # low = tough matchup (lots of touchbacks)

kick_team
LA     0.834926
NO     0.855400
LV     0.870852
TEN    0.870852
CLE    0.901756
WAS    0.924935
ATL    0.924935
NYJ    0.932661
ARI    0.932661
JAX    0.937811
CAR    0.959702
TB     0.963565
MIA    0.971291
NE     0.994101
GB     0.996186
MIN    1.002195
NYG    1.002195
DEN    1.004635
PHI    1.025374
SF     1.025374
LAC    1.032670
CHI    1.039199
HOU    1.046112
KC     1.048552
PIT    1.076452
BAL    1.079456
CIN    1.087182
DET    1.094908
DAL    1.110360
BUF    1.129066
SEA    1.137015
IND    1.187621
Name: is_return, dtype: float64

In [54]:
def score_return_line(yards, tds):
    """
        Calculate return fantasy points.
    """

    points = yards / 10.0 + 6.0 * tds
    for threshold in (100, 150, 200):
        points = points + 5.0 * (yards >= threshold)
    return points

def simulate_returner(exp_ko_returns, exp_pr_returns, pools, ko_multiplier=1.0,
                       n_sims=10_000, seed=0):
    """
        Play one player's game 10,000 times and see how fantasy scoring lands.

        Averages wouldn't accomodate Fantasy Scoring jumps, so we utilize simulation.
    """

    rng = np.random.default_rng(seed)
    yards = np.zeros(n_sims)
    tds = np.zeros(n_sims)

    # Scale expected ko returns by ko multiplier (opponent matchup)
    rates = {"kickoff": exp_ko_returns * ko_multiplier, "punt": exp_pr_returns}

    for kind, rate in rates.items():
        if rate <= 0 or kind not in pools:
            continue
        pool_y, pool_td = pools[kind]

        # Draw how many returns he gets per fake game
        #   Uses Poisson sample - standard way to randomize an event count
        n_ret = rng.poisson(rate, size=n_sims)
        total = int(n_ret.sum())
        if total == 0:
            continue

        # Select that many returns (as integers to grab from the pool)
        picks = rng.integers(0, len(pool_y), size=total)
        sim_id = np.repeat(np.arange(n_sims), n_ret)

        # Efficiennt way of summing values that share sim_id, faster than looping
        # Fill in yards/tds, for each sim, with the sum of the picked returns
        yards += np.bincount(sim_id, weights=pool_y[picks], minlength=n_sims)
        tds += np.bincount(sim_id, weights=pool_td[picks], minlength=n_sims)

    # Calculate fantasy points across sims
    points = score_return_line(yards, tds)

    # Return Simulation Summary
    return {
        "exp_points": points.mean(),
        "median_points": float(np.median(points)),
        "p10_points": float(np.percentile(points, 10)),
        "p90_points": float(np.percentile(points, 90)),
        "prob_100": float((yards >= 100).mean()),
        "prob_td": float((tds > 0).mean()),
    }


In [51]:
def build_leaderboard(usage, pools, opp_mult=None, opponents=None,
                      min_returns=1.0, n_sims=10_000):

    # Remove players that are expected less than 1 return
    q = usage[usage.exp_total_returns >= min_returns].copy()

    if opponents is not None and opp_mult is not None:
        q["opponent"] = q.return_team.map(opponents)
        q["ko_multiplier"] = q.opponent.map(opp_mult).fillna(1.0)
    else:
        q["opponent"] = None
        q["ko_multiplier"] = 1.0

    # Simulate Returner for each player in usage
    results = [
        simulate_returner(row.exp_ko_returns, row.exp_pr_returns, pools,
                          ko_multiplier=row.ko_multiplier, n_sims=n_sims, seed=i)
        for i, row in enumerate(q.itertuples())
    ]

    out = pd.concat([q.reset_index(drop=True), pd.DataFrame(results)], axis=1)
    return out.sort_values("exp_points", ascending=False).reset_index(drop=True)

def week_opponents(season, week):
    """
        Build a lookup of who each team plays in a given week
    """
    sched = nfl.load_schedules(seasons=[season]).to_pandas()
    g = sched[(sched.season == season) & (sched.week == week)]

    # Merge two dicts: home -> away, and away -> home.
    return {**dict(zip(g.home_team, g.away_team)),
            **dict(zip(g.away_team, g.home_team))}

In [57]:
opponents = week_opponents(2026, 1)
board = build_leaderboard(usage, pools, opp_mult, opponents=opponents)
board[["returner_name", "return_team", "opponent", "ko_multiplier", "exp_ko_returns",
       "exp_pr_returns", "exp_points", "p90_points", "prob_100"]].head(25)

,returner_name,return_team,opponent,ko_multiplier,exp_ko_returns,exp_pr_returns,exp_points,p90_points,prob_100
0,K.Turpin,DAL,NYG,1.002195,4.058824,0.588235,15.51431,28.8,0.5456
1,M.Price,MIN,GB,0.996186,3.352941,1.875000,14.69007,28.2,0.5041
2,C.Dike,TEN,NYJ,0.932661,3.647059,1.437500,14.07504,27.8,0.4783
3,J.Noel,HOU,BUF,1.129066,1.842105,1.947368,9.26310,19.0,0.2575
4,D.Duvernay,CHI,CAR,0.959702,2.368421,1.500000,9.25031,18.9,0.2621
5,C.Jones,CIN,TB,0.963565,2.470588,0.812500,8.58559,18.2,0.2289
6,T.Etienne,CAR,CHI,1.039199,1.888889,1.235294,7.62383,17.2,0.1819
7,M.Washington,MIA,LV,0.870852,2.117647,1.250000,7.28383,16.8,0.1694
8,K.Johnson,TB,CIN,1.087182,1.588235,1.529412,7.22380,16.7,0.1608
9,G.Dortch,ARI,LAC,1.032670,1.823529,1.066667,7.09549,16.5,0.1604
